# ula-gpu: MuJoCo Warp vs MJX benchmark (Colab GPU runtime)
Runtime > Change runtime type > a GPU (T4 works; A100/L4 better). Run cells top to bottom.

In [ ]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2
import sys, platform; print(sys.version, platform.platform())

## Install (same pins as docker/Dockerfile.gpu)
If Colab ships an incompatible jax already, the install below replaces it; restart the runtime only if pip says so.

In [ ]:
import sys; assert sys.version_info >= (3, 12), "jax 0.12.x needs Python >= 3.12 (Colab: pick a 3.12 runtime or relax the jax pin)"
!pip install -q "jax[cuda12]==0.12.1" "mujoco==3.11.0" "mujoco-mjx==3.11.0" "warp-lang==1.19.0" "mujoco_warp==3.11.0" pandas
# Pins = the versions validated in the CPU venv. mujoco, mujoco-mjx and mujoco_warp must stay on the SAME version.
# If the Colab driver is too old for CUDA 12 wheels, see docs/GPU_BENCH.md.

## Get the repo
**Option A** (clone). Replace the URL; a private repo needs a token in the URL: `https://<TOKEN>@github.com/<user>/ula-gpu.git`.

In [ ]:
import os
REPO_URL = "<REPO_URL>"   # <-- PLACEHOLDER: set this, or skip to option B
if REPO_URL.startswith("<"):
    print("REPO_URL not set: use option B")
elif not os.path.isdir("/content/ula-gpu"):
    !git clone --depth 1 {REPO_URL} /content/ula-gpu
else:
    print("already cloned")

**Option B** (fallback): upload `ula-gpu-bench.tar.gz` (made by `bench/make_colab_tarball.sh`) to /content, then run:

In [ ]:
import os
if not os.path.isdir("/content/ula-gpu"):
    assert os.path.exists("/content/ula-gpu-bench.tar.gz"), "upload ula-gpu-bench.tar.gz first"
    !tar xzf /content/ula-gpu-bench.tar.gz -C /content
!ls /content/ula-gpu/bench

## Run (outside the repo dir so MUJOCO_LOG.TXT does not land in it)

In [ ]:
!mkdir -p /content/run && cd /content/run && rm -f /content/ula-gpu/bench/results_colab.json && \
  python /content/ula-gpu/bench/mjx_bench.py --device gpu --num-envs 1024,4096,16384 --steps 200 \
  --results /content/ula-gpu/bench/results_colab.json

In [ ]:
!cd /content/run && python /content/ula-gpu/bench/warp_bench.py --device gpu --num-envs 1024,4096,16384 --steps 200 \
  --results /content/ula-gpu/bench/results_colab.json 2>&1 | grep -v '^Module'

## Table and save

In [ ]:
import json, pandas as pd
r = json.load(open("/content/ula-gpu/bench/results_colab.json"))
df = pd.DataFrame(r)
cols = [c for c in ["engine","variant","num_envs","steps_per_s","realtime_factor","compile_s","finite","gpu","error"] if c in df.columns]
print(df[cols].to_string(index=False, float_format=lambda x: f"{x:,.1f}"))
from google.colab import files; files.download("/content/ula-gpu/bench/results_colab.json")  # then put it in ~/Downloads